# Demo

Den här notebooken skapades som en snabb demonstration av vår bästa metod för kompetensextraktion.  
Den visar vår utgångspunkt (taxonomin), hur EntityRuler-mönstren skapas och ett visuellt exempel på kompetensmatchning i praktiken.

In [ ]:
import json, re, html
import pandas as pd
from src.models import build_nlp, entity_ruler_extract_skills

from IPython.display import HTML, display

with open("../data/skills_2511.json", encoding="utf-8") as f:
    taxonomi_df = pd.DataFrame(json.load(f))

## 1. Hur samlade vi kompetenser?

Vi började från taxonomin, där varje skill är kopplad till ett ID och en kategori.

In [ ]:
taxonomi_df.head()

Vi delade upp kategorierna mellan oss och samlade var och en hårda färdigheter för vår tilldelade kategori.  
Vi skrev in dem i tre olika grupper:  
- **Tier 1** är för enkla och entydiga ord och fraser som kan matchas oavsett case.  
- **Tier 2** är för korta och tvetydiga ord som måste matchas exakt.  
- **Tier 3** är för egna skills som kan skrivas och sägas på flera sätt.

## 2. Hur byggs regler för Entity Ruler?

En funktion använder kompetenserna som vi samlade för att automatiskt bygga EntityRuler mönstren när en nlp-pipeline skapas. 

In [ ]:
nlp = build_nlp()

ruler = nlp.get_pipe("entity_ruler")

print(len(ruler.patterns), "patterns in total\n")

for p in ruler.patterns[:3]:
    print(p)

## 3. Exempel på skill extraction

Vi tog en riktig annons för att visa hur komplex och ostrukturerad dem kan vara. 

Vi kör annonsen genom pipelinen, och de markerade ytorna visar vad som matchade rätt, vad som missades och vad som felaktigt förutspåddes.

In [ ]:
COLORS = {"tp": "#b7e4c7", "fp": "#ffb3b3", "fn": "#ffe08a"}

def review_ad(ad, gold, nlp):
    """green = correct, red = extra, yellow = missed. Compares matched text to gold text."""
    gold_cf = {g.casefold() for g in gold}
    doc = nlp(ad)

    # Green / red: what the entity ruler matched, using its exact positions
    spans = []
    for ent in doc.ents:
        kind = "tp" if ent.text.casefold() in gold_cf else "fp"
        spans.append((ent.start_char, ent.end_char, kind))

    # Yellow: gold words that the ruler never matched anywhere
    matched_cf = {ent.text.casefold() for ent in doc.ents}
    missed = [g for g in gold if g.casefold() not in matched_cf]
    if missed:
        pattern = re.compile(
            r"(?<!\w)(" + "|".join(map(re.escape, sorted(missed, key=len, reverse=True))) + r")(?!\w)",
            re.IGNORECASE,
        )
        for m in pattern.finditer(ad):
            spans.append((m.start(), m.end(), "fn"))

    spans.sort()
    parts, last = [], 0
    for start, end, kind in spans:
        if start < last:
            continue
        parts.append(html.escape(ad[last:start]))
        parts.append(f'<span style="background:{COLORS[kind]};color:black">{html.escape(ad[start:end])}</span>')
        last = end
    parts.append(html.escape(ad[last:]))

    display(HTML(
        f"<span style='background:{COLORS['tp']};color:black'>correct</span> "
        f"<span style='background:{COLORS['fp']};color:black'>extra</span> "
        f"<span style='background:{COLORS['fn']};color:black'>missed</span><br><br>"
        f"<div style='white-space:pre-wrap'>{''.join(parts)}</div>"
    ))

In [ ]:
ad = "I rollen som Data Scientist blir du en del av ett etablerat och samarbetsorienterat agilt team\nDu kommer till exempel att:\nUtveckla automatiserade mjukvarulösningar med hjälp av maskininlärning och optimeringstekniker\nHålla dig uppdaterad kring den senaste utvecklingen inom AI, maskininlärning, optimering och data science\nVI SÖKER DIG SOM:\nHar erfarenhet av Python och relevanta verktyg inom Data Science och Machine Learning\nHar god förståelse för Data Science-, AI- och Machine Learning-koncept\nTalar och skriver obehindrat på engelska\nErfarenhet av Kubernetes, OpenShift eller Azure\n"

real_skills = ["mjukvarulösningar", "maskininlärning", "AI", "Python", "verktyg inom Data Science och Machine Learning", "Machine Learning", "Data Science-, AI- och Machine Learning-koncept", "engelska", "Kubernetes", "OpenShift", "Azure",]
predicted_skills = entity_ruler_extract_skills(ad, nlp)

review_ad(ad, real_skills, nlp)